# Deepfake Detection Dataset 2026: Baseline Training

This notebook runs the same baseline pipeline as the command-line scripts: image download, ResNet-18 training, and held-out evaluation.

For a quick smoke test, set `config["download"]["limit"]` and `config["training"]["epochs"]` to small values before running the download and training cells.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from deepfake_detection.config import load_config, resolve_path

config_path = PROJECT_ROOT / "configs/default.yaml"
config = load_config(config_path)
config

In [ ]:
# Optional smoke-test override. Uncomment before downloading/training.
# config["download"]["limit"] = 100
# config["training"]["epochs"] = 1
# config["training"]["batch_size"] = 16

## Step 1: Download Images

This reads `FINAL_DATASET.csv`, creates/normalizes train-val-test splits, downloads image URLs, and writes `data/processed/metadata_with_paths.csv`.

In [ ]:
from deepfake_detection.download_images import download_images

metadata = download_images(config_path)
metadata.head()

In [ ]:
metadata["download_ok"].value_counts(dropna=False)

## Step 2: Train Baseline

The baseline is a binary ResNet-18 classifier with a single-logit output and `BCEWithLogitsLoss`.

In [ ]:
from deepfake_detection.train import train

result = train(config_path)
result

## Step 3: Evaluate

Evaluate the best validation-F1 checkpoint on the test split and write metrics to `reports/test_metrics.json`.

In [ ]:
from deepfake_detection.evaluate import evaluate

report = evaluate(config_path, split="test")
report["metrics"]

In [ ]:
import pandas as pd

pd.DataFrame(report["confusion_matrix"], index=["actual_fake", "actual_real"], columns=["pred_fake", "pred_real"])